# Connect Four endgames: everything the repository still owes, in one queue (una sola cola)

**What this notebook does.** It drains a list. Nothing on it re-reads weights that already exist: every memo-free read the repository owed is archived, so the list is training. The only 9 items are the arms at fourteen empty cells that no runtime has ever trained - 3 repairs against the published suite, each followed by its own read on that suite and on the blind one. No checkpoint has to travel inside: each arm fits its own base and its own repair from the code in the bundle.

**Why they are owed.** `tab:blind` reports 22 repaired arms and not one of them holds at `W=1` on a board-disjoint blind suite; that is the paper's scope result, and every one of them is read by this verifier rather than by the loop that certified it. 22 of them now carry the paired control - the same seed's *pre-repair* checkpoint, read once per suite - and that is every arm whose base weights survive on this disk: the 0 that ran before `c4_gru_base.pt` was ever saved could only be paired by fitting a base again, which is a training run and not a read. `sec:union` is fully measured - all 3 arms re-read on A, on B and over the eighty positions, their bases the same way - and a doubled sample certified nothing on its own. Fourteen empty cells is the deepest point of the ladder, and the arms it still owes cost hours each: a free CPU runtime takes them more cheaply than a laptop that is also doing something else.

**What you have to do.** Run the cells in order, and re-run the pump whenever the runtime is taken
away. The code and every root suite ride inside as a hashed bundle, so there is no GitHub sign-in, no token, nothing to upload and no Drive mount. Your browser gets up to 1 zip - one per section of the list, when
that section's last item lands - and nothing before, so a session that dies three quarters of the way
through still delivered what it did. Keep every zip and tell me where they landed.

In [ ]:
import os, pathlib, zipfile

try:
    from google.colab import files as _browser     # descarga directa en el navegador
except ImportError:
    _browser = None

CONTENT = pathlib.Path(os.environ.get('COLAB_CONTENT', '/content'))
SLUG = 'finishall'
_pushed = {}      # puerta -> firma de lo ya descargado, para no repetir la descarga

# las puertas de este notebook: los JSON que cada una espera enteros, y lo extra que lleva su
# zip. Un zip se construye SOLO cuando su puerta esta completa; mientras algo esta en vuelo
# esta celda no toca el disco, que es lo que le costo tiempo de reloj a la escalera pasada.
# globs: lo que un brazo produce fuera de data/ y que no existe hasta que el corre - y lo
# unico lugar del mundo donde va a existir.
GATES = {
    "end14": {
        "zip": "end14",
        "outs": [
            "data/c4_end14_holdoutB_s0.json",
            "data/c4_end14_holdoutB_s1.json",
            "data/c4_end14_holdoutB_s2.json",
            "data/c4_end14_s0.json",
            "data/c4_end14_s1.json",
            "data/c4_end14_s2.json",
            "data/c4_end14_verify_s0.json",
            "data/c4_end14_verify_s1.json",
            "data/c4_end14_verify_s2.json"
        ],
        "extra": [
            "data/c4_endgame_roots_14_40_s0.json",
            "data/c4_endgame_roots_14_40_holdoutB.json"
        ],
        "globs": [
            "runs/c4_end14_s*/c4_gru*.pt"
        ]
    }
}


def artefacts(gate):
    g = GATES[gate]
    found = [pathlib.Path(p) for p in g['outs'] if pathlib.Path(p).is_file()]
    found += [pathlib.Path('runs', '_colab_' + pathlib.Path(p).name[:-5] + '.log')
              for p in g['outs'] if pathlib.Path('runs', '_colab_' +
              pathlib.Path(p).name[:-5] + '.log').is_file()]
    found += [pathlib.Path(p) for p in g['extra'] if pathlib.Path(p).is_file()]
    for pat in g['globs']:
        found += [p for p in sorted(pathlib.Path('.').glob(pat)) if p.is_file()]
    return sorted(set(found))


def _complete(gate):
    return all(pathlib.Path(p).is_file() for p in GATES[gate]['outs'])


def snapshot():
    """Entregar una puerta cuando su ultimo job aterrizo; a medias, nunca.

    La unica puerta de este notebook es end14 con 9 salidas: no se abre hasta que su ultima salida esta en disco - los JSON de cada repair y sus dos lecturas, los pesos que producen y las dos suites - asi que ningun zip se construye mientras un brazo entrena. La celda ship() del final es la salida de emergencia cuando la sesion se acaba.
    """
    return [a for a in [_push(g) if _complete(g) else None for g in GATES] if a]


def ship():
    """La salida de emergencia: entrega lo que haya, aunque una puerta este a medias."""
    return [a for a in [_push(g) for g in GATES] if a]


def _push(gate):
    found = artefacts(gate)
    sig = {p.as_posix(): p.stat().st_size for p in found if p.suffix != '.log'}
    if not found or sig == _pushed.get(gate):
        return None
    CONTENT.mkdir(parents=True, exist_ok=True)
    archive = CONTENT / ('c4_%s_%s_results.zip' % (SLUG, GATES[gate]['zip']))
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for p in found:
            z.write(p, arcname=p.as_posix())
    _pushed[gate] = sig
    size = archive.stat().st_size / 2 ** 20
    if _browser is None:
        print('%d artefactos -> %s (%.2f MB), sin Colab' % (len(found), archive, size))
    else:
        print('%d artefactos -> el navegador descarga %s (%.2f MB)'
              % (len(found), archive.name, size), flush=True)
        try:
            _browser.download(str(archive))
        except Exception as exc:
            # la descarga es el canal de salida, no el experimento: si el navegador no
            # está acá, la cola sigue viva y el zip espera en /content
            print('  AVISO: no se pudo enviar al navegador (%s); el zip está en %s'
                  % (exc, archive))
    return archive


print('sin Drive: hasta %d %s al navegador, uno por seccion de la cola vaciada (o con la '
      'celda ship())' % (len(GATES), 'zip' if len(GATES) == 1 else 'zips'))

## 0. Nothing to set up

The cell titled *The bundle* holds a base64 zip of the four modules this queue runs and every root
suite in the repository, with a sha256 beside each file. The setup cell re-computes those hashes and
unpacks them into a working copy under `/content`. A bundle that does not match - edited inside the
notebook, or older than the code - is refused, and the cell falls back to cloning the repository.

In [ ]:
# The bundle: this notebook carries the code it runs, so a runtime needs nothing
# from GitHub - no authorisation, no token, no upload. Built by
# colab/build_bundle.py from commit 0be3a4d. Every file's sha256 is here and
# the setup cell re-computes them, so a hand-edited or stale bundle falls back to
# the clone instead of running modified code.
import base64 as _b64, hashlib as _hl, io, zipfile

BUNDLE_COMMIT = '0be3a4d'
BUNDLE_SHA256 = 'f6e0fd8416f2856a8179502fe42ce8fa8714d43bf11a04ef73f43179085cf6df'
BUNDLE_FILES = {
    "src/c4_game.py": "f9d0dece4145b957b461e59427faea7ebcdafedf4f28823facac1236f7bd55c4",
    "src/c4_repair.py": "13323ae5495a35100c69599fe71a00300a8a9097934952b866e46e8e968feccc",
    "src/c4_endgame.py": "d40a600ca87e1d38de42ea6ab3afea8d78cef3b1f0bedee31d9d118070ec9204",
    "src/c4_endgame_verify.py": "47009f16a1649db948b0f8fcafce4d688864c8cc10c1762fd6d89d549a92ade7",
    "data/c4_end10_s1.json": "4189ccc5599c3495a11899368b7b0647699080654d3801ae4cb5359f15d7ef63",
    "data/c4_endgame_roots_10_40_holdoutB.json": "274c14a6a2d402defa9fba26373797cf44d9d97cec0ef490a825de3a062e4a4a",
    "data/c4_endgame_roots_10_40_s0.json": "3079b51610d93dd269e5104c539b63782f34f3a980199867cd6e5bab75055a00",
    "data/c4_endgame_roots_12_40_holdoutB.json": "e671bf8c7cd7b6df32643a9653c529c1cade2588226eb36597eb5ace6b786e5a",
    "data/c4_endgame_roots_12_40_s0.json": "3ef57d6feccd9a492fa99c3258b38458ace19205b7d9aebc215d39548f0d7c10",
    "data/c4_endgame_roots_14_40_holdoutB.json": "3530ecc16d6dc659763f0cc21b6ff50199d9ae404a685cd6692594e35db99e61",
    "data/c4_endgame_roots_14_40_s0.json": "006d1ced87af527ecdbb7bd8e8e8e9a2980e0316b809c1a4d20cfc8b3cc4a18a",
    "data/c4_endgame_roots_8_40_holdoutB.json": "a829bf5c47e30a0ebe671c9d4e3c1c5103dbb90b97d41e0f0cd396165dcc1082",
    "data/c4_endgame_roots_8_40_s0.json": "d3c67bde3ce07ca68384b453d58574798f9425bd2e5f2aa2118be07ac65fccc4",
    "data/c4_endgame_roots_10_80_AplusB.json": "b9ce69507eb481da8d82ed5cbb0dc88b16ba58ea22c7a36d4daeb2741120d879",
    "data/c4_endgame_roots_8_80_AplusB.json": "68e02bedc82fe8837175fdc854ab2fca9f438ee33c6850febe67bfeb4c296a24"
}
BUNDLE_B64 = (
    '...'
    'iQQAABlXBQAAAA=='
)

_archive = _b64.b64decode(BUNDLE_B64)
assert _hl.sha256(_archive).hexdigest() == BUNDLE_SHA256, 'the bundle is corrupt'
_zf = zipfile.ZipFile(io.BytesIO(_archive))
assert sorted(_zf.namelist()) == sorted(BUNDLE_FILES), 'the bundle holds other files than it claims'
for _n in _zf.namelist():
    assert _hl.sha256(_zf.read(_n)).hexdigest() == BUNDLE_FILES[_n], 'bundle was modified: ' + _n
print('this notebook carries its own code: commit %s, %d files, %.0f KB unpacked,'
      ' %.0f KB inside the notebook - no GitHub sign-in needed'
      % (BUNDLE_COMMIT, len(BUNDLE_FILES),
        sum(z.file_size for z in _zf.infolist()) / 1024, len(_archive) / 1024))


In [ ]:
import base64, hashlib, io, os, pathlib, re, shutil, subprocess, sys, zipfile

URL = 'https://github.com/starlyn2010/fragility-paper.git'
REPO_SLUG = 'starlyn2010/fragility-paper'
MARKER = pathlib.Path('src') / 'c4_endgame.py'
CONTENT = pathlib.Path(os.environ.get('COLAB_CONTENT', '/content'))
TOKEN = ''  # optional: a read-only GitHub token, only if you prefer it over route (1)


def _find_repo():
    """Any directory under /content that already holds the repo, wherever it was put."""
    seen = [pathlib.Path.cwd(), CONTENT / 'fragility-paper']
    if CONTENT.is_dir():
        seen += sorted(CONTENT.glob('*/'))
    return next((c for c in seen if (c / MARKER).exists()), None)


def _unzip_repo():
    """A repo archive in Drive or uploaded to the runtime counts as a copy of the repo.

    Two layouts have to work: an archive made by GitHub, which wraps everything in one
    folder, and `git archive`, which stores the paths flat from the repo root. So the
    candidate is the extraction directory itself as well as each top-level entry.
    """
    if not CONTENT.is_dir():
        return None
    for z in sorted(CONTENT.glob("**/*fragility-paper*.zip")):
        with zipfile.ZipFile(z) as fh:
            names = fh.namelist()
            fh.extractall(z.parent)
        tops = sorted({n.split("/")[0] for n in names if "/" in n})
        roots = [z.parent] + [z.parent / t for t in tops]
        hit = next((r for r in roots if (r / MARKER).exists()), None)
        if hit:
            print("unzipped repo archive ->", hit)
            return hit
        print("unzipped", z.name, "but it has no", MARKER)
    return None


def _unbundle():
    """Rebuild a working copy of the repository from the cell that carries it.

    The bundle cell above holds the modules this queue runs, the ten-cell root suite
    and the archived seed-1 arm its controls-from arms read, so a runtime with the
    notebook needs nothing else: no GitHub authorisation, no token, no upload. Nothing is trusted on the way in - the archive's
    own hash and every file's hash recorded in the bundle are recomputed here, and any
    mismatch returns None so bootstrap falls through to the clone instead of running code
    that was edited inside the notebook.
    """
    if "BUNDLE_B64" not in globals():
        return None
    archive = base64.b64decode(BUNDLE_B64)
    if hashlib.sha256(archive).hexdigest() != BUNDLE_SHA256:
        print("the notebook's bundle does not match its own hash; ignoring it")
        return None
    target = CONTENT / "fragility-paper"
    with zipfile.ZipFile(io.BytesIO(archive)) as fh:
        names = sorted(fh.namelist())
        if names != sorted(BUNDLE_FILES):
            print("the notebook's bundle holds other files than it claims; ignoring it")
            return None
        for name in names:
            blob = fh.read(name)
            if hashlib.sha256(blob).hexdigest() != BUNDLE_FILES[name]:
                print("the notebook's bundle was modified:", name)
                return None
            out = target / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(blob)
    print("built a working copy of the repo from the notebook itself ->", target)
    return target if (target / MARKER).exists() else None


def _gh_ready():
    """True if the runtime's gh CLI is installed and already holds credentials.

    Colab ships gh preinstalled, so this needs no pip install.
    """
    if shutil.which('gh') is None:
        return False
    return subprocess.run(['gh', 'auth', 'status'], capture_output=True,
                          text=True).returncode == 0


def _gh_device_login():
    """Ask GitHub to authorise THIS runtime, with a code you type on any device.

    Nothing is pasted into the notebook and no token is stored in it: the credential
    lives in this runtime and dies with it. Two details matter on Colab. There is no
    browser to open, so BROWSER is set to a command that does nothing. And gh paints
    its progress with ANSI cursor moves, which the notebook output frame can choke on,
    so the stream is cleaned line by line and printed as it arrives. The cell blocks
    while you authorise, which is the point: copy the one-time code, type it at
    github.com/login/device from any device, and this cell carries on by itself.
    """
    bs = chr(92)
    ansi = re.compile(chr(27) + bs + "[" + "[0-9;?<>!]*" + "[A-Za-z~]"
                      + "|" + chr(27) + "[()]" + "[A-Za-z]"
                      + "|" + chr(27) + "[=<>]")
    print('logging this runtime in to GitHub (device flow).', flush=True)
    p = subprocess.Popen(['gh', 'auth', 'login', '--hostname', 'github.com',
                          '--git-protocol', 'https', '--web'],
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'BROWSER': 'echo'})
    try:
        p.stdin.write(chr(10))
        p.stdin.flush()
    except Exception:
        pass
    for raw in iter(p.stdout.readline, ''):
        line = "".join(c for c in ansi.sub("", raw) if c == chr(10) or ord(c) >= 32)
        if line.strip():
            print(line.strip(), flush=True)
    p.wait()
    return _gh_ready()


def _clone(url):
    target = CONTENT / 'fragility-paper'
    if target.exists():
        shutil.rmtree(target)
    if shutil.which('gh') and _gh_ready():
        print("cloning with the runtime's stored GitHub credentials")
        r = subprocess.run(['gh', 'repo', 'clone', REPO_SLUG, str(target),
                            '--', '--depth', '1'], capture_output=True, text=True)
        if not r.returncode and (target / MARKER).exists():
            return target
        sys.stderr.write((r.stderr or r.stdout).strip()[-400:] + '\n')
        return None
    if TOKEN and '://' in url:
        head, rest = url.split('://', 1)
        url = head + '://x-access-token:' + TOKEN + '@' + rest
    print('fetching the repo ->', url)
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(target)],
                       capture_output=True, text=True, env=env)
    if r.returncode:
        sys.stderr.write((r.stderr or r.stdout).strip()[-400:] + '\n')
        return None
    return target if (target / MARKER).exists() else None


def bootstrap(url=URL, ask_gh=True):
    """Find the repo, get it if this runtime has none, and make `src` importable.

    A Colab kernel leaves /content on sys.path and %cd does not change it, so the repo
    root has to be inserted by hand, and so does its src directory: the modules
    under src import each other by their flat names (import c4_game), so a cell
    that reaches for src.c4_game loads a SECOND copy of the module and then
    configures the copy the sampler is not using.
    """
    repo = _find_repo() or _unbundle() or _unzip_repo() or _clone(url)
    if repo is None and ask_gh and not _gh_ready():
        if shutil.which('gh'):
            _gh_device_login()
            repo = _clone(url)
    if repo is None:
        raise SystemExit(
            'no copy of the repo could be made, and the bundle that normally removes '
            'that need was not usable. Run the cell titled "The bundle" just above this '
            'one, then this one; it rebuilds the working copy from the notebook itself '
            'and needs no GitHub sign-in at all. If that cell is missing, this notebook '
            'is an old copy - re-upload colab/c4_finish_all.ipynb. As a last resort '
            'this cell can still fetch the repo: let it sign the runtime in with the '
            'GitHub device flow (a one-time code you type at github.com/login/device, '
            'dies with the runtime), or upload a zip of the repo through the file '
            'sidebar and re-run.')
    os.chdir(repo)
    for entry in (str(repo), str((repo / 'src').resolve())):
        if entry not in sys.path:
            sys.path.insert(0, entry)
    return repo.resolve()


REPO = bootstrap()
print('repo at', REPO, '| on sys.path:', str(REPO) in sys.path)
!python -c "import torch, platform; print(platform.python_version(), torch.__version__)"

## 1. How much machine do we actually have

A read is one single-threaded recursion and an arm is one recursion plus a small GRU refit, so the
queue holds two jobs and gives each a thread. This is the same capacity cell the training notebooks
run, and nothing here is configured by hand.

In [ ]:
import os

def quota_cores():
    try:                                       # cgroup v2
        quota, period = open('/sys/fs/cgroup/cpu.max').read().split()
        if quota != 'max':
            return max(1, int(float(quota) / int(period)))
    except Exception:
        pass
    try:                                       # cgroup v1
        quota = int(open('/sys/fs/cgroup/cpu/cpu.cfs_quota_us').read())
        if quota > 0:
            return max(1, quota // int(open('/sys/fs/cgroup/cpu/cpu.cfs_period_us').read()))
    except Exception:
        pass
    return os.cpu_count() or 2

def avail_gb():
    try:                                       # `/proc` is a Linux thing: a reviewer
        for line in open('/proc/meminfo'):     # running this in local Jupyter on a
            if line.startswith('MemAvailable'):
                return int(line.split()[1]) / 2 ** 20
    except OSError:                            # Mac has none
        return None

CORES, RAM = quota_cores(), avail_gb()
# An arm is one single-threaded recursion plus a small GRU refit; budget ~2.5 GB per
# concurrent process and never oversubscribe the CPUs. Where RAM cannot be read the
# queue holds at two arms: an out-of-memory kill loses the whole run, while an idle
# core only costs time.
MAX_PAR = max(1, min(CORES, 2 if RAM is None else int(RAM // 2.5)))
THREADS = max(1, CORES // MAX_PAR)
print(f'usable cores {CORES}, available RAM ' +
      ('not reported' if RAM is None else f'{RAM:.1f} GB'))
print(f'=> {MAX_PAR} arms at a time, --threads {THREADS} each')

## 2. The weights this notebook carries

None, and the section says so rather than printing an empty table. A read of weights that already
exist is what needs a checkpoint to travel inside as base64, because `*.pt` files are not in git - and
every read of that kind the repository owed is archived now. What this document holds is training: each
arm fits its own base, saves `c4_gru_base.pt` next to `c4_gru.pt`, and is then re-read by its own two
verifications inside this same queue, on the weights it just wrote. The cells of this section are the
machinery that would lay the weights down, kept in place so the document has one shape whether or not
it carries bytes, and the last cell of the section still refuses the document if a checkpoint is listed
without the cell of bytes that belongs to it.

In [ ]:
try:
    REPO
except NameError:
    raise SystemExit('run the section 0 setup cell first: it unpacks the bundle and makes '
                     'the directory the weights are written into')
import base64, hashlib, pathlib

# checkpoint -> su sha256, su tamano y el brazo cuyo certificado debe reproducir. Los bytes
# vienen despues, una celda por checkpoint, para que ningun celda pese megabytes.
WEIGHTS = {}
_seen = []


def _put(rel, parts):
    """Escribir un checkpoint del notebook en el disco del runtime, despues de comprobarlo."""
    if rel not in WEIGHTS:
        raise SystemExit('%s: este notebook no cola ninguna lectura de ese checkpoint, asi que '
                         'no hay certificado con el comprobarlo; no se escribe' % rel)
    meta = WEIGHTS[rel]
    blob = base64.b64decode(''.join(parts))
    got = hashlib.sha256(blob).hexdigest()
    if got != meta['sha256']:
        raise SystemExit('%s: el base64 de esta celda no hash a lo que el notebook anuncia (%s), '
                         'así que no es el checkpoint del brazo %s; no se lee nada'
                         % (rel, got[:12], meta['arm']))
    if len(blob) != meta['bytes']:
        raise SystemExit('%s: %d bytes esperados, %d en la celda' %
                         (rel, meta['bytes'], len(blob)))
    p = pathlib.Path(rel)
    if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == got:
        print('  ya está   %s (%s del brazo %s)' % (rel, got[:12], meta['arm']))
    else:
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(blob)
        print('  escrito   %s | %s | brazo %s, %d vacías, certificado en %s'
              % (rel, got[:12], meta['arm'], meta['E'], meta['arm_json']))
    _seen.append(rel)

In [ ]:
faltan = sorted(set(WEIGHTS) - set(_seen))
if faltan or len(_seen) != len(WEIGHTS):
    raise SystemExit('faltan las celdas de bytes de %d checkpoints (%s): sin ellos las lecturas '
                     'que los necesitan esperan para siempre, y un zip parcial no paga la deuda'
                     % (len(faltan), ', '.join(faltan[:4])))
if not WEIGHTS:
    print('0 checkpoints dentro: ninguna lectura de esta cola pesa pesos ya entrenados; los '
          'unicos archivos .pt que existiran aqui son los que entrenen los brazos de catorce '
          'vacias, y salen en el zip end14')
else:
    print('%d checkpoints en el runtime | %.2f MB en total | %d celdas de pesos corre'
          % (len(WEIGHTS), sum(m['bytes'] for m in WEIGHTS.values()) / 2 ** 20, len(_seen)))

## 3. The suites the queue is pointed at

The published suite A and the blind suite B, at 14 empty cells.  All of them travel in the bundle, and this cell does not just open them: it refuses
a suite whose positions are not unfinished draws at the depth it claims, refuses A and B of a depth if
they share a board (a holdout that overlaps the training set is not a holdout). No union suite is queued, because every arm certified over `A + B` has already been read that way, so this cell has no `U` to decompose.

In [ ]:
import json, pathlib

import c4_game as g
g.configure(6, 7, 4)

# las suites que esta cola punta, tal como las trae el bundle, con su profundidad: la
# publicatoria A, la ciega B y - para los brazos certificados sobre las dos - la union U
SUITES = {
    "A14": "data/c4_endgame_roots_14_40_s0.json",
    "B14": "data/c4_endgame_roots_14_40_holdoutB.json"
}
SUITES_E = {
    "A14": 14,
    "B14": 14
}


def _boards(roots):
    return {tuple(r['board']) for r in roots}


_cache = {}
for key, path in sorted(SUITES.items()):
    p = pathlib.Path(path)
    if not p.is_file():
        raise SystemExit('%s is not in the runtime: the suite a read is pointed at has to be '
                         'the committed one, so this notebook stops rather than sampling a '
                         'different set of positions' % path)
    roots = json.loads(p.read_text(encoding='utf-8'))
    _cache[key] = roots
    E = SUITES_E[key]
    want = 80 if key[0] == 'U' else 40
    assert len(roots) == want, 'suite %s holds %d positions, not %d' % (key, len(roots), want)
    for r in roots:
        assert len(r['board']) == g.N, 'suite %s: not a full 6x7 board' % key
        empties = r['board'].count(0)      # una celda vacía es 0; g.EMPTY es el tablero vacío
        assert empties == E, ('suite %s: a position with %d empty cells is not a %d-empty '
                              'endgame' % (key, empties, E))
        assert not g.terminal(tuple(r['board'])), 'suite %s: a finished position' % key
        assert r['value'] == 0, 'suite %s: a root the learner cannot draw' % key
    print('suite %-3s | %2d posiciones | todas unfinished y de valor 0 | %2d vacías | %s'
          % (key, len(roots), E, path))

for _E in sorted(set(SUITES_E.values())):
    a, b = 'A%d' % _E, 'B%d' % _E
    if a in _cache and b in _cache:
        shared = _boards(_cache[a]) & _boards(_cache[b])
        assert not shared, ('suite B of %d empties is not a holdout: %d positions are also in '
                            'suite A, so a blind read of them measures nothing' % (_E, len(shared)))
        print('A%d y B%d comparten 0 posiciones' % (_E, _E))
    u = 'U%d' % _E
    if u in _cache:
        assert _cache[u] == _cache[a] + _cache[b], (
            'suite U of %d empties is not suite A followed by suite B, so a certificate over it '
            'is not a certificate over both halves' % _E)
        print('U%d es exactamente A seguida de B: %d + %d = %d'
              % (_E, len(_cache[a]), len(_cache[b]), len(_cache[u])))

print('las suites están validadas; cada lectura espera a la suya antes de arrancar')

## 4. The queue

9 jobs, two at a time: 3 repairs at fourteen empty cells, each followed by its own two reads - the repaired policy re-read on the suite it certified over, and on the blind one. Every job waits for the file before it: a read of a new arm waits for that arm's repair JSON and for the weights it just trained, so a queue launched on a runtime where the pump was interrupted waits instead of quietly reading a file that is not there.  Nothing here re-runs a result the repository already has: the plan below was
generated from the JSONs on disk, and every one of these output files is absent from it.

The only zip this document opens is the fourteen-empty one, and it opens when all 9 of its outputs are on disk - 3 repairs and their 6 verifications - so nothing is packaged while an arm trains, which is the one thing the last ladder run paid wall time for.

In [ ]:
import json, os, pathlib, shutil, subprocess

# una lectura = un checkpoint, una suite, un JSON de salida; un brazo = un repair y sus dos
# lecturas. Generado por colab/build_reads_notebooks.py a partir de colab/reads_plan.py, sobre
# los archivos que faltaban en data/ en el momento de escribir este notebook.
READS = []
ARMS = [
    {
        "arm": "c4_end14_s0",
        "kind": "arm-repair",
        "which": "A",
        "E": 14,
        "seed": 0,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s0/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_s0.json",
        "needs": [],
        "arm_json": "data/c4_end14_s0.json"
    },
    {
        "arm": "c4_end14_s0",
        "kind": "arm-read-on-A",
        "which": "A",
        "E": 14,
        "seed": 0,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s0/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_verify_s0.json",
        "needs": [
            "data/c4_end14_s0.json",
            "runs/c4_end14_s0/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s0.json"
    },
    {
        "arm": "c4_end14_s0",
        "kind": "arm-read-on-B",
        "which": "B",
        "E": 14,
        "seed": 0,
        "key": "B14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s0/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_holdoutB.json",
        "out": "data/c4_end14_holdoutB_s0.json",
        "needs": [
            "data/c4_end14_s0.json",
            "runs/c4_end14_s0/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s0.json"
    },
    {
        "arm": "c4_end14_s1",
        "kind": "arm-repair",
        "which": "A",
        "E": 14,
        "seed": 1,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s1/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_s1.json",
        "needs": [],
        "arm_json": "data/c4_end14_s1.json"
    },
    {
        "arm": "c4_end14_s1",
        "kind": "arm-read-on-A",
        "which": "A",
        "E": 14,
        "seed": 1,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s1/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_verify_s1.json",
        "needs": [
            "data/c4_end14_s1.json",
            "runs/c4_end14_s1/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s1.json"
    },
    {
        "arm": "c4_end14_s1",
        "kind": "arm-read-on-B",
        "which": "B",
        "E": 14,
        "seed": 1,
        "key": "B14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s1/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_holdoutB.json",
        "out": "data/c4_end14_holdoutB_s1.json",
        "needs": [
            "data/c4_end14_s1.json",
            "runs/c4_end14_s1/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s1.json"
    },
    {
        "arm": "c4_end14_s2",
        "kind": "arm-repair",
        "which": "A",
        "E": 14,
        "seed": 2,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s2/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_s2.json",
        "needs": [],
        "arm_json": "data/c4_end14_s2.json"
    },
    {
        "arm": "c4_end14_s2",
        "kind": "arm-read-on-A",
        "which": "A",
        "E": 14,
        "seed": 2,
        "key": "A14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s2/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_s0.json",
        "out": "data/c4_end14_verify_s2.json",
        "needs": [
            "data/c4_end14_s2.json",
            "runs/c4_end14_s2/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s2.json"
    },
    {
        "arm": "c4_end14_s2",
        "kind": "arm-read-on-B",
        "which": "B",
        "E": 14,
        "seed": 2,
        "key": "B14",
        "gate": "end14",
        "ckpt": "runs/c4_end14_s2/c4_gru.pt",
        "suite_file": "data/c4_endgame_roots_14_40_holdoutB.json",
        "out": "data/c4_end14_holdoutB_s2.json",
        "needs": [
            "data/c4_end14_s2.json",
            "runs/c4_end14_s2/c4_gru.pt"
        ],
        "arm_json": "data/c4_end14_s2.json"
    }
]
BUDGET = {
    "agg_repeat": 60,
    "stop": "both",
    "max_iters": 24,
    "epochs": 200,
    "lr": 0.002,
    "hidden": 128,
    "emb": 16,
    "branch": 3,
    "roots": 40,
    "root_value": 0
}   # el presupuesto con que se publico la profundidad de la que heredan


def _suite(row):
    """La suite que la seccion 3 valido, y el chequeo de que la cola apunta exactamente a ella."""
    p = SUITES[row['key']]
    assert p == row['suite_file'], ('%s: la cola apunta a %s pero la celda de suites valido %s, '
                                    'o sea que esta lectura correria sobre posiciones que nadie '
                                    'comprobo' % (row['out'], row['suite_file'], p))
    return p


def repair(a):
    """El unico comando de src/c4_endgame.py del notebook: el brazo nuevo, con su repair."""
    E, s = a['E'], a['seed']
    return dict(name='repair %d vacias s%d' % (E, s), out=a['out'], needs=[_suite(a)],
                gate=a['gate'],
                cmd=['python', 'src/c4_endgame.py', '--rows', '6', '--cols', '7',
                     '--win', '4', '--empties', str(E), '--roots', str(BUDGET['roots']),
                     '--root-value', str(BUDGET['root_value']),
                     '--branch', str(BUDGET['branch']), '--hidden', str(BUDGET['hidden']),
                     '--emb', str(BUDGET['emb']), '--epochs', str(BUDGET['epochs']),
                     '--lr', str(BUDGET['lr']), '--agg-repeat', str(BUDGET['agg_repeat']),
                     '--stop', str(BUDGET['stop']), '--max-iters', str(BUDGET['max_iters']),
                     '--roots-file', _suite(a), '--threads', str(THREADS), '--seed', str(s),
                     '--no-controls', '--ckpt-out', 'runs/c4_end%d_s%d' % (E, s),
                     '--out', a['out']])


def verify(src_row):
    """Una lectura memo-free: src/c4_endgame_verify.py sobre un checkpoint y una suite.

    Espera el checkpoint y la suite siempre, y en los brazos nuevos tambien el JSON del repair
    que todavia no existe: una lectura de unos pesos a medio escribir no es una lectura.
    """
    r = src_row
    waits = list(r['needs'])
    p = _suite(r)
    if p not in waits:
        waits.append(p)
    if r['kind'].startswith('arm-'):
        name = '%s nuevo' % r['arm']
    else:
        name = '%s %s' % (r['arm'], 'repaired' if 'repaired' in r['kind'] else 'base')
    return dict(name='%s on %s' % (name, r['which']), out=r['out'], needs=waits,
                gate=r['gate'],
                cmd=['python', 'src/c4_endgame_verify.py', '--ckpt', r['ckpt'],
                     '--roots', p, '--hidden', '128', '--emb', '16',
                     '--orders', '120', '--games', '40', '--out', r['out']])


JOBS = ([verify(r) for r in READS]
        + [repair(a) if a['kind'] == 'arm-repair' else verify(a) for a in ARMS])
for j in JOBS:
    j['log'] = 'runs/_colab_' + j['out'].split('/')[-1][:-5] + '.log'

_suites = sorted({row['suite_file'] for row in READS + ARMS})
print('%d lecturas + %d brazos nuevos = %d jobs, %d checkpoints dentro del notebook, %d suites, '
      '%d corriendo a la vez' % (len(READS), len({a['arm'] for a in ARMS}), len(JOBS),
                                 len(WEIGHTS), len(_suites), MAX_PAR))
for _s in _suites:
    print('   suite', _s)
for a in ARMS:
    if a['kind'] == 'arm-repair':
        print('   repair %s: --agg-repeat %s --stop %s --max-iters %s sobre %s'
              % (a['arm'], BUDGET['agg_repeat'], BUDGET['stop'], BUDGET['max_iters'],
                 a['suite_file']))

### Monitor

One line per job: running, done, or waiting on the file that has to exist first. A `ready` line means
a slot is free and `tick()` has not been called since.

In [ ]:
try:
    REPO = bootstrap()
except NameError:
    raise SystemExit('run the section 0 setup cell first - it clones the repo and '
                     'puts its root on sys.path')
import json, os, pathlib, shutil, subprocess, sys, time

def pids(needle):
    """Pids whose command line contains `needle`; callers pass 'out <path>',
    without the leading dashes, because pgrep would read --out as its own option."""
    r = subprocess.run(['pgrep', '-f', needle], capture_output=True, text=True)
    return [int(x) for x in r.stdout.split()]

def tick():
    live = sum(1 for j in JOBS if pids('out ' + j['out']))
    for j in JOBS:
        if os.path.exists(j['out']) or pids('out ' + j['out']):
            continue
        if not all(os.path.exists(p) for p in j['needs']):
            continue
        if live >= MAX_PAR:
            break
        # a clone has runs/ because runs/*.sh is tracked, but the working copy
        # built from this notebook's own bundle ships src/ and data/ only
        pathlib.Path(j['log']).parent.mkdir(parents=True, exist_ok=True)
        fh = open(j['log'], 'w')
        subprocess.Popen(j['cmd'], stdout=fh, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, cwd=str(REPO))
        live += 1
        print('started', j['name'], '->', j['log'], flush=True)
    snapshot()
    left = [j['name'] for j in JOBS if not os.path.exists(j['out'])]
    print(f'{len(JOBS) - len(left)}/{len(JOBS)} JSONs on disk | {live} running | '
          f'{len(left)} to go')

tick()

In [ ]:
for j in JOBS:
    if os.path.exists(j['out']):
        print(f'  done  {j["name"]}')
    elif pids('out ' + j['out']):
        tail = ''
        if os.path.exists(j['log']):
            lines = [l for l in open(j['log']).read().splitlines() if l.strip()]
            tail = lines[-1] if lines else ''
        print(f'  RUN   {j["name"]} | {tail[:104]}')
    elif not all(os.path.exists(p) for p in j['needs']):
        print(f'  wait  {j["name"]}')
    else:
        print(f'  ready {j["name"]} (a slot is free - run tick() again)')

if not [j for j in JOBS if not os.path.exists(j['out'])]:
    print('queue empty')

In [ ]:
# Hands-free: pump the queue until it empties, then hand control back.

# Running this cell is activity, so the runtime is not idle-disconnected while it
# works. tick() hands every finished arm to the browser, so an interrupt or a
# 12-hour session cap costs the arms in flight and nothing already done: re-run
# the cell and the queue resumes where it stopped.
MAX_MINUTES = 700
t_start = time.time()
while True:
    tick()
    for j in JOBS:
        if pids('out ' + j['out']) and os.path.exists(j['log']):
            _l = [x for x in open(j['log']).read().splitlines() if x.strip()]
            print('  RUN   %s | %s' % (j['name'], _l[-1][:104] if _l else ''))
    left = [j['name'] for j in JOBS if not os.path.exists(j['out'])]
    if not left:
        print('queue empty - every arm is on disk and in the browser'); break
    if (time.time() - t_start) / 60 > MAX_MINUTES:
        print(f'{len(left)} arms to go after {MAX_MINUTES} min - run this cell again'); break
    time.sleep(120)

The pump keeps the runtime alive and returns by itself after 700 minutes, so the session is released before Colab takes it away. Interrupt it whenever you like: a job that finished is on disk, and re-running the cell resumes where it stopped. Only the job in flight restarts, and at this depth every job in the list is a repair that takes hours, so a session that ends mid-repair loses that repair and nothing else.

Every pass it also prints a `RUN` line per job still in flight, with that job's last log line. This
is the only way to tell a working repair from a stalled one: an arm writes its JSON on the final
round, so the number of finished jobs stays exactly where it was for hours while the rounds keep
coming. A counter that does not move and a `RUN` line whose round number does not move are different
problems; a counter that does not move and a round number that does is just what fourteen empty
cells look like.

## 5. Read the result, and check it against the arm that made these weights

2 things are verified rather than printed. First, a read on the suite its arm certified over has to equal the certificate in that arm's own JSON - worst case, best case, mean, roots drawn - because that is the only way to tell "the memo-free recursion agrees" from "the notebook read somebody else's checkpoint"; for the arms this notebook trains, the comparison runs against the repair JSON the runtime itself just wrote. Second, the counts the tables publish, with Wilson intervals: certifications on the blind suite, reported separately for the arms repaired against suite A and the new fourteen-empty ones - the union population prints nothing here, because no arm certified over `A + B` is left in this queue. The cell refuses the first failure and names the read that broke it.

In [ ]:
import json, math, os, pathlib

# Cada linea de abajo es una lectura memo-free de src/c4_endgame_verify.py, y lo que cada brazo
# ya afirmó sobre la suite que certificó, copiado de su propio JSON por
# colab/build_reads_notebooks.py. Una lectura memo-free de esos mismos pesos tiene que dar lo
# mismo; si no coincide, el notebook leyó otros pesos. Los brazos que este notebook entrena no
# tienen copia: la expectation se lee del JSON que el propio runtime acaba de escribir.
EXPECTED = {}


def wilson(k, n, z=1.96):
    """95% interval for a rate measured on n arms; (nan, nan) while there are none."""
    if n == 0:
        return float('nan'), float('nan')
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


KEYS = ('worst_draw_rate', 'best_draw_rate', 'random_draw_rate_exact', 'n_roots_drawn_worst')
ALL = READS + [a for a in ARMS if a['kind'] != 'arm-repair']


def expectation(row):
    """Lo que el brazo ya afirmo de si mismo, en la suite que el archivo dice que certifico."""
    repaired = row['kind'].startswith(('arm-read-on-', 'repaired-on-', 'union-repaired-'))
    e = EXPECTED.get(row['arm'], {}).get('repaired' if repaired else 'base')
    if e:
        return e
    p = pathlib.Path(row.get('arm_json', ''))
    if not p.is_file():
        return None
    d = json.loads(p.read_text(encoding='utf-8'))
    roots_file = d['args']['roots_file'].replace("\\", "/")
    return {'cert': d['repair']['final'] if repaired else d['base']['certificate'],
            'suite': ('U' if 'AplusB' in roots_file else
                      'B' if 'holdoutB' in roots_file else 'A')}


read = {}
for r in ALL:
    if os.path.exists(r['out']):
        read[(r['arm'], r['kind'], r['which'])] = json.loads(pathlib.Path(r['out'])
                                                             .read_text(encoding='utf-8'))

left = [r for r in ALL if not os.path.exists(r['out'])]
print('=== %d de %d jobs en disco, %d por hacer ===' % (len(read), len(ALL), len(left)))

fail = []
for r in ALL:
    v = read.get((r['arm'], r['kind'], r['which']))
    if v is None:
        continue
    exp = expectation(r)
    tag = '%-16s %-22s on %s' % (r['arm'], r['kind'], r['which'])
    line = ('W %.3f B %.3f D %.4f | raices a tablas %d/%d | ordenes %d (%.3f-%.3f) | '
            'adaptativo %.3f-%.3f'
            % (v['worst_draw_rate'], v['best_draw_rate'], v['random_draw_rate_exact'],
               v['n_roots_drawn_worst'], v['roots'], v['deterministic_orders'],
               v['order_min'], v['order_max'], v['adaptive_min'], v['adaptive_mean']))
    if exp and exp['suite'] == r['which']:
        c = exp['cert']
        same = all(abs(c[k] - v[k]) < 1e-9 for k in KEYS)
        line += ' | el loop decia %s -> %s' % (
            ', '.join('%.3f' % c[k] if isinstance(c[k], float) else str(c[k]) for k in KEYS),
            'COINCIDE' if same else 'NO COINCIDE')
        if not same:
            fail.append(tag + ': estos pesos no son los del brazo ' + r['arm'])
    print('  %s %s' % (tag, line))

# la identidad sobre la que se apoya sec:union: dos suites desjuntas, W del todo = min de las
# mitades, las raices suman y el promedio es el promedio de los promedios
for arm in sorted({r['arm'] for r in READS}):
    got = {}
    for (a, kind, w), v in read.items():
        if a == arm and kind.startswith('union-'):
            tier = 'repaired' if 'repaired' in kind else 'base'
            got.setdefault(tier, {})[w] = v
    for tier, half in sorted(got.items()):
        if not all(t in half for t in ('A', 'B', 'U')):
            continue
        u, a, b = half['U'], half['A'], half['B']
        ok = (abs(u['worst_draw_rate'] - min(a['worst_draw_rate'], b['worst_draw_rate'])) < 1e-12
              and abs(u['random_draw_rate_exact'] -
                      (a['random_draw_rate_exact'] + b['random_draw_rate_exact']) / 2) < 1e-12
              and u['n_roots_drawn_worst'] ==
                      a['n_roots_drawn_worst'] + b['n_roots_drawn_worst']
              and u['roots'] == a['roots'] + b['roots'])
        print('  %-16s %s identidad W_U = min(W_A, W_B): %s  (W %.2f/%.2f/%.2f, raices '
              '%d+%d=%d)' % (arm, tier, 'OK' if ok else 'ROTO', a['worst_draw_rate'],
                              b['worst_draw_rate'], u['worst_draw_rate'],
                              a['n_roots_drawn_worst'], b['n_roots_drawn_worst'],
                              u['n_roots_drawn_worst']))
        if not ok:
            fail.append('%s %s: la union no es el minimo de sus dos mitades' % (arm, tier))

# lo que el propio loop certifico en los brazos que este notebook entrena, junto a sus lecturas
for a in [x for x in ARMS if x['kind'] == 'arm-repair']:
    if not os.path.exists(a['out']):
        continue
    d = json.loads(pathlib.Path(a['out']).read_text(encoding='utf-8'))
    fin = d['repair']['final']
    print('  %-16s repair: W %.2f B %.2f D %.3f | rondas %d | historias %d | %.0f s'
          % (a['arm'], fin['worst_draw_rate'], fin['best_draw_rate'],
             fin['random_draw_rate_exact'], len(d['repair']['log']) - 1,
             len(d['repair']['aggregate']), d['repair']['elapsed_s']))

# el recuento que las tablas publican: certificados sobre la suite ciega, separando las tres
# poblaciones - reparados contra A (tab:blind), contra la union (sec:union) y los de catorce
for label, pref in (('tab:blind: reparados contra la suite A', 'repaired-on-B'),
                    ('sec:union: reparados contra la union A+B', 'union-repaired-on-B'),
                    ('nuevos a catorce vacias: reparados contra A', 'arm-read-on-B')):
    rows = [(arm, v) for (arm, kind, w), v in sorted(read.items()) if kind == pref]
    if not rows:
        continue
    k = sum(1 for _a, v in rows if v['worst_draw_rate'] == 1.0)
    lo, hi = wilson(k, len(rows))
    print('  %s: %d de %d llegan a W=1 ciego [%.2f, %.2f]' % (label, k, len(rows), lo, hi))
    for arm, v in rows:
        print('     %-16s W %.2f | B %.3f | D %.3f | %d/%d raices a tablas'
              % (arm, v['worst_draw_rate'], v['best_draw_rate'],
                 v['random_draw_rate_exact'], v['n_roots_drawn_worst'], v['roots']))

if fail:
    print()
    for f in fail:
        print('  NO VALE: ' + f)
    raise SystemExit('una lectura no reproduce a la que el brazo certifico: los pesos de este '
                     'notebook no son los de ese brazo, o las suites no son las dos mitades '
                     'que se afirma que son')
if left:
    print()
    print('  faltan %d jobs; corre la celda pump otra vez' % len(left))
print('descarga final:')
for a in snapshot() or []:
    print('  ', a.name)

## 6. Ship what exists

The queue hands a zip to your browser when the last item of its section lands. This cell does not
wait: it zips whatever is on disk right now, section by section, and sends it over. Run it as the last
cell of a session, and run it *before* you close the tab if Colab warns that the runtime is ending -
without it, whatever finished after the last download dies with that runtime. Re-running is safe: a
zip is only rebuilt when its contents changed.

In [ ]:
suelta = ship()
print('%d zip%s al navegador%s' % (len(suelta), '' if len(suelta) == 1 else 's',
      '' if suelta else ': nada nuevo, cada seccion ya fue descargada'))
for a in suelta:
    print('  ', a.name)

## 7. When the queue is empty

The 9 arm jobs - 3 repairs, 6 verifications - their logs and the fourteen-empty checkpoints they trained are in the `end14` zip your browser took, and those weights are the only copy that will ever exist, because `*.pt` is not in git, so that zip matters more than the numbers in it. Send them to me and I will do the rest: the numbers go into
`tab:blind`, `sec:union` and the two generators that print them, and the weights stay where they are.
Please do not re-run the queue afterwards - it overwrites nothing, and it would spend hours the paper
does not need.